## 基线 Transformer + 后处理（point-adjust / 平滑 / 连续段约束）

说明：本 notebook **不改动你的基线实现**，而是在其输出的 `test_point_scores` 基础上增加后处理与事件级评价。

输出：
- Raw（纯 point-wise 阈值）P/R/F1
- Point-adjust（事件级）P/R/F1
- 平滑+连续段约束后的 P/R/F1（可选再做 point-adjust）


In [ ]:
from pathlib import Path
import os
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

# ============ 可复现 ============
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ============ 路径（与 数据处理.ipynb 保持一致） ============
train_path = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_14days_new.csv")
test_path  = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_attackdataLABLE.csv")

assert os.path.exists(train_path), f"not found: {train_path}"
assert os.path.exists(test_path), f"not found: {test_path}"

train_df = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_raw:", test_raw.shape)

# ============ 处理测试集表头与标签 ============
test_raw.columns = test_raw.iloc[0]
test_raw = test_raw.iloc[1:].reset_index(drop=True)
if len(test_raw) >= 2:
    test_raw = test_raw.iloc[:-2]

label_col = "Attack LABLE (1:No Attack, -1:Attack)"
assert label_col in test_raw.columns, f"label column not found: {label_col}"

test_labels = test_raw[label_col].replace({1: 0, -1: 1}).astype(int).to_numpy()
test_df = test_raw.drop(columns=[label_col])

# drop meta
for c in ["Row ", "Date ", "Time"]:
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])

for c in ["Row", "Date", "Time"]:
    if c in train_df.columns:
        train_df = train_df.drop(columns=[c])

# 数值化
train_df = train_df.dropna(how="all").astype(float)
test_df = test_df.astype(float)

# ============ NaN 清洗 ============
all_nan_cols_train = train_df.columns[train_df.isna().all(axis=0)]
all_nan_cols_test  = test_df.columns[test_df.isna().all(axis=0)]
all_nan_cols = sorted(list(set(all_nan_cols_train).union(set(all_nan_cols_test))))

if len(all_nan_cols) > 0:
    print("Drop all-NaN columns:", len(all_nan_cols))
    train_df = train_df.drop(columns=all_nan_cols)
    test_df  = test_df.drop(columns=all_nan_cols)

train_median = train_df.median()
train_df = train_df.fillna(train_median)
test_df  = test_df.fillna(train_median)

print("train_df clean:", train_df.shape)
print("test_df  clean:", test_df.shape)
print("test_labels:", test_labels.shape, "anomaly count:", int(test_labels.sum()))

# ============ 标准化（只用训练集 fit） ============
scaler = StandardScaler()
train_data = scaler.fit_transform(train_df.values)
test_data  = scaler.transform(test_df.values)

train_data = np.nan_to_num(train_data, nan=0.0, posinf=0.0, neginf=0.0)
test_data  = np.nan_to_num(test_data,  nan=0.0, posinf=0.0, neginf=0.0)

D = train_data.shape[1]
print("feature dim D =", D)

# ============ Dataset（带窗口起点索引，用于时间点级聚合） ============
class SlidingWindowWithIndexDataset(Dataset):
    def __init__(self, data: np.ndarray, window_size: int, stride: int = 1):
        assert stride >= 1
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size
        self.stride = stride

    def __len__(self):
        max_start = len(self.data) - self.window_size
        if max_start < 0:
            return 0
        return max_start // self.stride + 1

    def __getitem__(self, i: int):
        start = i * self.stride
        x = self.data[start : start + self.window_size]
        return x, start

window_size = 100
batch_size = 64
train_stride = 2  # 训练加速（你可改 1/2/5/10）

train_ds_train = SlidingWindowWithIndexDataset(train_data, window_size, stride=train_stride)
train_ds_eval  = SlidingWindowWithIndexDataset(train_data, window_size, stride=1)
test_ds        = SlidingWindowWithIndexDataset(test_data,  window_size, stride=1)

train_loader = DataLoader(train_ds_train, batch_size=batch_size, shuffle=True, drop_last=False)
train_eval_loader = DataLoader(train_ds_eval, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

xb, ib = next(iter(train_loader))
print("batch x:", xb.shape, "batch idx:", ib.shape)

# ============ 模型：Transformer Encoder-Decoder 重构 ============
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

class TransformerReconstruction(nn.Module):
    def __init__(self, input_dim: int, d_model: int = 256, nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3,
                 dim_feedforward: int = 512, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)
        self.output_proj = nn.Linear(d_model, input_dim)

    def forward(self, src: torch.Tensor):
        src_emb = self.pos_enc(self.input_proj(src))
        tgt_emb = src_emb  # teacher forcing
        memory = self.encoder(src_emb)
        dec_out = self.decoder(tgt_emb, memory)
        return self.output_proj(dec_out)

model = TransformerReconstruction(input_dim=D).to(device)

# ============ 训练 ============
lr = 1e-4
weight_decay = 1e-4
epochs = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
recon_criterion = nn.MSELoss(reduction="mean")

def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    n = 0
    for x, _idx in loader:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        x_hat = model(x)
        loss = recon_criterion(x_hat, x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += float(loss.item()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)

for ep in range(1, epochs + 1):
    loss = train_one_epoch(model, train_loader)
    scheduler.step()
    print(f"Epoch {ep:03d} | recon_loss={loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}")

# ============ 时间点级 anomaly score（窗口误差摊回全序列） ============
@torch.no_grad()
def compute_point_scores(model, loader, seq_len: int):
    model.eval()
    score_sum = np.zeros(seq_len, dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int64)

    for x, idx in loader:
        x = x.to(device)
        idx = idx.numpy()
        x_hat = model(x)
        err = (x_hat - x).pow(2).mean(dim=2).detach().cpu().numpy()  # [B,L]
        B, L = err.shape
        for b in range(B):
            s = int(idx[b])
            e = s + L
            score_sum[s:e] += err[b]
            count[s:e] += 1

    return score_sum / np.maximum(count, 1)

train_point_scores = compute_point_scores(model, train_eval_loader, seq_len=len(train_data))
test_point_scores  = compute_point_scores(model, test_loader,      seq_len=len(test_data))

print("train_point_scores:", train_point_scores.shape)
print("test_point_scores :", test_point_scores.shape)
                   
print("baseline ready.")


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def prf1(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return float(p), float(r), float(f1)

def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """经典 point-adjust：对每个真实异常连续段，只要命中任意一点，就把整段预测置为 1。"""
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            # [i, j) 是一个异常段
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x
    k = int(k)
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(k, dtype=np.float64) / k
    return np.convolve(xp, kernel, mode="valid")

def apply_segment_constraints(pred: np.ndarray, min_len: int = 10, merge_gap: int = 5) -> np.ndarray:
    """去毛刺：短于 min_len 的异常段删除；间隔 <= merge_gap 的异常段合并。"""
    pred = pred.astype(int)
    n = len(pred)
    # 先找所有异常段
    segments = []
    i = 0
    while i < n:
        if pred[i] == 1:
            j = i
            while j < n and pred[j] == 1:
                j += 1
            segments.append([i, j])
            i = j
        else:
            i += 1

    # 合并 gap 小的段
    merged = []
    for s, e in segments:
        if not merged:
            merged.append([s, e])
        else:
            ps, pe = merged[-1]
            if s - pe <= merge_gap:
                merged[-1][1] = e
            else:
                merged.append([s, e])

    # 删除短段
    out = np.zeros(n, dtype=int)
    for s, e in merged:
        if (e - s) >= min_len:
            out[s:e] = 1
    return out


In [ ]:
# ============ 选择阈值（固定一个，不做 for 扫描） ============
# 根据搜索结果，q=0.905 附近点级 F1 最好
quantile = 0.905
threshold = float(np.quantile(train_point_scores, quantile))

raw_pred = (test_point_scores > threshold).astype(int)
raw_p, raw_r, raw_f1 = prf1(test_labels, raw_pred)

raw_pred_pa = point_adjust(test_labels, raw_pred)
pa_p, pa_r, pa_f1 = prf1(test_labels, raw_pred_pa)

print(f"threshold@q={quantile}: {threshold:.6f}")
print(f"Raw     P={raw_p:.4f} R={raw_r:.4f} F1={raw_f1:.4f}")
print(f"PA(adjust) P={pa_p:.4f} R={pa_r:.4f} F1={pa_f1:.4f}")


In [ ]:
# ============ 平滑 + 连续段约束（减少毛刺误报） ============
# 参数建议：
# - smooth_k: 7~31 之间试（越大越平滑）
# - min_len : 最短异常持续点数（比如 10/20/30）
# - merge_gap: 段间距 <= merge_gap 会合并

smooth_k = 1   # 搜索结果：不平滑 + 较短最小段 + 较大 merge_gap 效果最好
min_len = 10
merge_gap = 20

smoothed_scores = moving_average(test_point_scores.astype(np.float64), smooth_k)
sm_pred = (smoothed_scores > threshold).astype(int)
sm_pred = apply_segment_constraints(sm_pred, min_len=min_len, merge_gap=merge_gap)

sm_p, sm_r, sm_f1 = prf1(test_labels, sm_pred)
sm_pred_pa = point_adjust(test_labels, sm_pred)
sm_pa_p, sm_pa_r, sm_pa_f1 = prf1(test_labels, sm_pred_pa)

print(f"Smooth(k={smooth_k}) + seg(min_len={min_len}, gap={merge_gap})")
print(f"Post    P={sm_p:.4f} R={sm_r:.4f} F1={sm_f1:.4f}")
print(f"Post+PA P={sm_pa_p:.4f} R={sm_pa_r:.4f} F1={sm_pa_f1:.4f}")


In [ ]:
# ============ 可视化（对比 raw vs post） ============
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 4))
plt.plot(test_point_scores, linewidth=1, alpha=0.7, label="raw score")
plt.plot(smoothed_scores, linewidth=1, alpha=0.9, label=f"smoothed(k={smooth_k})")
plt.axhline(threshold, color="r", linestyle="--", linewidth=1, label=f"threshold(q={quantile})")

anom = np.where(test_labels == 1)[0]
if len(anom) > 0:
    plt.scatter(anom, smoothed_scores[anom], s=3, alpha=0.25, label="attack points")

plt.title("WADI test anomaly score (raw vs smoothed)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============ 保存分数与标签（下次无需重训，直接加载即可） ============
import os
import numpy as np

# 保存到当前 notebook 所在目录
out_path = "baseline_scores_and_labels.npz"

np.savez(
    out_path,
    train_point_scores=train_point_scores,
    test_point_scores=test_point_scores,
    test_labels=test_labels,
)

print("Saved to:", os.path.abspath(out_path))
print("Keys:", list(np.load(out_path).keys()))
